# Merge Stage 1 -- Build Panel: Macro Daily

## Purpose
Combines all daily and weekly macro-level cleaned datasets into a single panel keyed on `date`, aligned to the CRSP trading calendar. This produces the macro-level daily panel (Panel C).

## Sources (All Cleaned)

### Daily (Left Join on Date)
- `Data/Data_Collection/Cleaned/02_FRED/fred_daily_clean.parquet` -- interest rates, credit spreads, commodities, yield curve factors
- `Data/Data_Collection/Cleaned/09_Macro_Daily_Monthly_WRDS/macro_daily_clean.parquet` -- VIX family, Fama-French 5 + momentum, risk-free rate, FX rates, world index returns
- `Data/Data_Collection/Cleaned/11_VIX_SKEW/vix_skew_combined_clean.parquet` -- VIX futures term structure, CBOE SKEW

### Weekly (merge_asof Backward to Carry Forward)
- `Data/Data_Collection/Cleaned/05_CFTC/cftc_clean.parquet` -- futures positioning (uses `available_date` for point-in-time)
- `Data/Data_Collection/Cleaned/03_AAII_Sentiment/aaii_sentiment_clean.parquet` -- bullish/bearish/neutral survey (published Thursday)
- `Data/Data_Collection/Cleaned/02_FRED/fred_weekly_clean.parquet` -- three sub-groups with different publication lags

## Key Design Decisions
- **Trading calendar from Panel A:** the set of unique dates from the CRSP-based stock daily panel defines which days exist in this panel. This ensures macro and stock panels are perfectly aligned.
- **Forward-fill (limit=5 days)** applied to daily sources for holiday gaps (FRED/WRDS holidays that are CRSP trading days).
- **Weekly sources use `merge_asof(direction='backward')`** to carry each weekly value forward to every trading day until the next observation arrives.
- **CFTC uses `available_date` (Monday release), not `date` (Tuesday position).** Merging on the position date would cause 4-day look-ahead bias.
- **FRED weekly dates are reference dates, not publication dates.** Per-series shifts are applied to align to actual publication day before the `merge_asof` join.
- **No winsorisation, no z-standardisation** -- deferred to later stages.

## Merge Logic

### Step 1: Establish Trading Calendar
Unique dates are extracted from the already-built `panel_stock_daily.parquet` (Panel A). This guarantees the macro panel covers exactly the same trading days as the stock panel.

### Step 2: Join FRED Daily
Left-joined on `date` onto the trading calendar. Forward-fill with limit=5 applied to cover holiday gaps where FRED did not report but CRSP traded.

### Step 3: Join WRDS Macro Daily
Left-joined on `date`. Column name conflicts with FRED daily are resolved by adding a `wrds_` prefix. Forward-fill with limit=5 for holiday gaps.

### Step 4: Join VIX Futures + CBOE SKEW
Left-joined on `date`. Column name conflicts resolved with `vs_` prefix. Forward-fill with limit=5 for holiday gaps only -- the structural NaN from VIX futures pre-launch (Jan--Mar 2004) and SKEW warmup periods are not filled.

### Step 5: Join CFTC (Weekly, Point-in-Time)
CFTC data is merged using `available_date` (the Monday release date, 6 days after the Tuesday position date) as the merge key, not the report `date`. This prevents look-ahead bias. `merge_asof(direction='backward')` carries each weekly observation forward until the next release. CFTC data starts June 2006, so earlier trading days have NaN.

### Step 6: Join AAII Sentiment (Weekly)
AAII sentiment survey (published Thursday) is merged via `merge_asof(direction='backward')`. Each trading day gets the most recent AAII observation.

### Step 7: Join FRED Weekly (Per-Series Publication Lag)
FRED weekly data is split into three groups based on their Federal Reserve release schedule, and each group's reference dates are shifted to publication dates before merging:

- **Department of Labor claims** (`initial_claims`, `continued_claims`): Saturday reference shifted +5 days to Thursday publication
- **Federal Reserve H.4.1** (`fed_assets`, `tga`, `reserves`): Wednesday reference shifted +1 day to Thursday publication
- **Federal Reserve H.8** (`bank_credit`, `ci_loans`): Wednesday reference shifted +9 days to Friday of the following week publication

Each group is merged independently via `merge_asof(direction='backward')` to avoid the sparse-outer-join trap where NaN from one group would overwrite another.

### Step 8: Compute Derived Factors
- **`vix_futures_basis`** = `vix_fut_front` - `vix` (front-month futures minus VIX spot). Positive = futures premium (normal), negative = spot premium (fear). Both components are now available in the same panel.

### Step 9: Final Column Inventory
All factors categorised by source: FRED daily, WRDS macro daily, VIX/SKEW, CFTC, AAII, FRED weekly, and derived.

### Step 10: Validation
- No duplicate dates
- Row count, date range, total columns
- NaN summary by source after forward-fill
- Per-column remaining NaN listing (top 25)
- FRED weekly health check: verifies publication lag shifts by confirming first valid dates are plausible
- Day-of-week distribution (should be weekdays only)
- Sample rows printed including a mid-2020 row to verify all sources are populated

## Output
`Data/Data_Collection/Final/Stage_1/panel_macro_daily.parquet` -- keyed on `date`, containing all daily and weekly macro-level factors from FRED, WRDS, VIX/SKEW, CFTC, AAII, and derived

In [5]:
# %% [markdown]
# # Merge Pipeline — Notebook 03: Build Panel C (Macro Daily)
#
# Combines all daily and weekly macro-level datasets into a single panel
# keyed on (date), aligned to the trading calendar.
#
# Daily sources (left join on date):
#   - FRED daily — interest rates, credit spreads, etc.
#   - WRDS macro daily — VIX family, FF5+UMD, FX rates, world index returns
#   - VIX futures + CBOE SKEW — term structure, tail risk
#
# Weekly sources (merge_asof backward to carry forward):
#   - CFTC — futures positioning (uses available_date for point-in-time)
#   - AAII sentiment — bullish/bearish/neutral survey (published Thursday)
#   - FRED weekly — three sub-groups with different publication lags:
#       DoL claims (Sat ref → Thu pub, +5d)
#       Fed H.4.1 (Wed ref → Thu pub, +1d)
#       Fed H.8 (Wed ref → Fri next week pub, +9d)
#
# Trading calendar is established from Panel A (CRSP trading days).
#
# KEY DESIGN DECISIONS:
#   - Forward-fill applied here for holiday gaps (limit=5 days).
#   - Weekly sources use merge_asof(direction='backward') to carry each
#     weekly value forward to every trading day until the next observation.
#   - CFTC uses available_date (Monday release) not date (Tuesday position).
#   - FRED weekly dates are reference dates, not publication dates. Per-series
#     shifts applied to align to actual publication day (+1/+5/+9 days).
#   - No winsorisation, no z-standardisation — those happen in Step 3.
#
# Output: Data/Data_Collection/Final/Stage_1/panel_macro_daily.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
CLEANED = Path('../../../Data/Data_Collection/Cleaned')
PANEL_A = Path('../../../Data/Data_Collection/Final/Stage_1/panel_stock_daily.parquet')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_1')
OUT_DIR.mkdir(parents=True, exist_ok=True)

FRED_DAILY_PATH  = CLEANED / '02_FRED' / 'fred_daily_clean.parquet'
MACRO_DAILY_PATH = CLEANED / '09_Macro_Daily_Monthly_WRDS' / 'macro_daily_clean.parquet'
VIX_SKEW_PATH    = CLEANED / '11_VIX_SKEW' / 'vix_skew_combined_clean.parquet'
CFTC_PATH        = CLEANED / '05_CFTC' / 'cftc_clean.parquet'
AAII_PATH        = CLEANED / '03_AAII_Sentiment' / 'aaii_sentiment_clean.parquet'
FRED_WEEKLY_PATH = CLEANED / '02_FRED' / 'fred_weekly_clean.parquet'

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: ESTABLISH TRADING CALENDAR FROM PANEL A
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: ESTABLISH TRADING CALENDAR")
print("=" * 90)

panel_a_dates = pd.read_parquet(PANEL_A, columns=['date'])
calendar = pd.DataFrame({
    'date': pd.to_datetime(panel_a_dates['date'].unique())
}).sort_values('date').reset_index(drop=True)
del panel_a_dates

print(f"\n  Trading calendar: {len(calendar):,} days")
print(f"  Date range: {calendar['date'].min().date()} → {calendar['date'].max().date()}")
print(f"  Weekends: {(calendar['date'].dt.dayofweek >= 5).sum()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: JOIN FRED DAILY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: JOIN FRED DAILY")
print("=" * 90)

fred_d = pd.read_parquet(FRED_DAILY_PATH)
fred_d['date'] = pd.to_datetime(fred_d['date'])
fred_d_factors = [c for c in fred_d.columns if c != 'date']

print(f"\n  FRED daily: {len(fred_d):,} rows × {len(fred_d_factors)} factors")

panel = calendar.merge(fred_d, on='date', how='left')
del fred_d

n_matched = panel[fred_d_factors[0]].notna().sum()
print(f"  Matched: {n_matched:,} / {len(calendar):,} ({n_matched/len(calendar)*100:.1f}%)")

# Forward-fill holiday gaps (FRED holidays that are CRSP trading days)
pre_nan = panel[fred_d_factors].isna().sum().sum()
panel[fred_d_factors] = panel[fred_d_factors].ffill(limit=5)
post_nan = panel[fred_d_factors].isna().sum().sum()
print(f"  Forward-fill (limit=5): {pre_nan:,} → {post_nan:,} NaN "
      f"(filled {pre_nan - post_nan:,})")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: JOIN WRDS MACRO DAILY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 3: JOIN WRDS MACRO DAILY")
print("=" * 90)

macro_d = pd.read_parquet(MACRO_DAILY_PATH)
macro_d['date'] = pd.to_datetime(macro_d['date'])
macro_d_factors = [c for c in macro_d.columns if c != 'date']

print(f"\n  WRDS macro daily: {len(macro_d):,} rows × {len(macro_d_factors)} factors")

# Check for column name conflicts
overlap = set(fred_d_factors) & set(macro_d_factors)
if overlap:
    print(f"  ⚠ Column overlap with FRED daily: {overlap}")
    macro_d = macro_d.rename(columns={c: f'wrds_{c}' for c in overlap})
    macro_d_factors = [c for c in macro_d.columns if c != 'date']

n_before = len(panel)
panel = panel.merge(macro_d, on='date', how='left')
del macro_d
assert len(panel) == n_before, f"Row explosion: {n_before} → {len(panel)}"

n_matched = panel[macro_d_factors[0]].notna().sum()
print(f"  Matched: {n_matched:,} / {len(panel):,} ({n_matched/len(panel)*100:.1f}%)")

# Forward-fill holiday gaps
pre_nan = panel[macro_d_factors].isna().sum().sum()
panel[macro_d_factors] = panel[macro_d_factors].ffill(limit=5)
post_nan = panel[macro_d_factors].isna().sum().sum()
print(f"  Forward-fill (limit=5): {pre_nan:,} → {post_nan:,} NaN "
      f"(filled {pre_nan - post_nan:,})")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: JOIN VIX FUTURES + CBOE SKEW
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: JOIN VIX FUTURES + CBOE SKEW")
print("=" * 90)

vix_skew = pd.read_parquet(VIX_SKEW_PATH)
vix_skew['date'] = pd.to_datetime(vix_skew['date'])
vix_skew_factors = [c for c in vix_skew.columns if c != 'date']

print(f"\n  VIX/SKEW: {len(vix_skew):,} rows × {len(vix_skew_factors)} factors")

# Check for column name conflicts
all_existing = set(panel.columns) - {'date'}
overlap = all_existing & set(vix_skew_factors)
if overlap:
    print(f"  ⚠ Column overlap: {overlap}")
    vix_skew = vix_skew.rename(columns={c: f'vs_{c}' for c in overlap})
    vix_skew_factors = [c for c in vix_skew.columns if c != 'date']

n_before = len(panel)
panel = panel.merge(vix_skew, on='date', how='left')
del vix_skew
assert len(panel) == n_before, f"Row explosion: {n_before} → {len(panel)}"

n_matched = panel[vix_skew_factors[0]].notna().sum()
print(f"  Matched: {n_matched:,} / {len(panel):,} ({n_matched/len(panel)*100:.1f}%)")

# Forward-fill holiday gaps only (NOT the structural Jan-Mar 2004 VIX futures gap)
pre_nan = panel[vix_skew_factors].isna().sum().sum()
panel[vix_skew_factors] = panel[vix_skew_factors].ffill(limit=5)
post_nan = panel[vix_skew_factors].isna().sum().sum()
print(f"  Forward-fill (limit=5): {pre_nan:,} → {post_nan:,} NaN "
      f"(filled {pre_nan - post_nan:,})")
print(f"  (Remaining NaN is structural: VIX futures launch Mar 2004, SKEW warmup)")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: JOIN CFTC (WEEKLY — POINT-IN-TIME)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: JOIN CFTC (WEEKLY — POINT-IN-TIME)")
print("=" * 90)

cftc = pd.read_parquet(CFTC_PATH)
cftc['date'] = pd.to_datetime(cftc['date'])

# CFTC must have available_date (Monday release, 6 days after Tuesday position).
# Merging on 'date' directly would cause 4-day look-ahead bias.
assert 'available_date' in cftc.columns, (
    "FATAL: 'available_date' missing from CFTC data. "
    "Merging on 'date' would cause 4-day look-ahead bias."
)
cftc['available_date'] = pd.to_datetime(cftc['available_date'])
print(f"\n  Using available_date for point-in-time merge (6-day lag)")

cftc_factors = [c for c in cftc.columns if c not in ['date', 'available_date']]
print(f"  CFTC: {len(cftc):,} rows × {len(cftc_factors)} factors")
print(f"  Date range: {cftc['available_date'].min().date()} → "
      f"{cftc['available_date'].max().date()}")

# Prepare for merge_asof: use available_date as the merge key
cftc_for_merge = cftc[['available_date'] + cftc_factors].copy()
cftc_for_merge = cftc_for_merge.rename(columns={'available_date': 'date'})
cftc_for_merge = cftc_for_merge.sort_values('date').reset_index(drop=True)

# Check column name conflicts
all_existing = set(panel.columns) - {'date'}
overlap = all_existing & set(cftc_factors)
if overlap:
    print(f"  ⚠ Column overlap: {overlap}")
    rename_map = {c: f'cftc_{c}' for c in overlap}
    cftc_for_merge = cftc_for_merge.rename(columns=rename_map)
    cftc_factors = [rename_map.get(c, c) for c in cftc_factors]

# merge_asof: each trading day gets the most recent CFTC observation
panel = panel.sort_values('date').reset_index(drop=True)

n_before = len(panel)
panel = pd.merge_asof(
    panel,
    cftc_for_merge,
    on='date',
    direction='backward'
)
del cftc, cftc_for_merge
assert len(panel) == n_before, f"Row explosion: {n_before} → {len(panel)}"

n_matched = panel[cftc_factors[0]].notna().sum()
print(f"  Matched: {n_matched:,} / {len(panel):,} ({n_matched/len(panel)*100:.1f}%)")

# Check: CFTC starts June 2006, so early dates should be NaN
first_valid = panel[panel[cftc_factors[0]].notna()]['date'].min()
print(f"  First valid CFTC date: {first_valid.date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: JOIN AAII SENTIMENT (WEEKLY)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 6: JOIN AAII SENTIMENT (WEEKLY)")
print("=" * 90)

aaii = pd.read_parquet(AAII_PATH)
aaii['date'] = pd.to_datetime(aaii['date'])
aaii_factors = [c for c in aaii.columns if c != 'date']

print(f"\n  AAII: {len(aaii):,} rows × {len(aaii_factors)} factors")
print(f"  Date day-of-week: {aaii['date'].dt.day_name().value_counts().to_dict()}")
print(f"  (Should be ~100% Thursday — AAII published Thursday)")

# Check column name conflicts
all_existing = set(panel.columns) - {'date'}
overlap = all_existing & set(aaii_factors)
if overlap:
    print(f"  ⚠ Column overlap: {overlap}")
    rename_map = {c: f'aaii_{c}' for c in overlap}
    aaii = aaii.rename(columns=rename_map)
    aaii_factors = [rename_map.get(c, c) for c in aaii_factors]

aaii = aaii.sort_values('date').reset_index(drop=True)
panel = panel.sort_values('date').reset_index(drop=True)

n_before = len(panel)
panel = pd.merge_asof(
    panel,
    aaii,
    on='date',
    direction='backward'
)
del aaii
assert len(panel) == n_before, f"Row explosion: {n_before} → {len(panel)}"

n_matched = panel[aaii_factors[0]].notna().sum()
print(f"  Matched: {n_matched:,} / {len(panel):,} ({n_matched/len(panel)*100:.1f}%)")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: JOIN FRED WEEKLY (PER-SERIES PUBLICATION LAG)
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 7: JOIN FRED WEEKLY (PER-SERIES PUBLICATION LAG)")
print("=" * 90)

fred_w = pd.read_parquet(FRED_WEEKLY_PATH)
fred_w['date'] = pd.to_datetime(fred_w['date'])
print(f"\n  FRED weekly: {len(fred_w):,} rows × {fred_w.shape[1] - 1} factors")
print(f"  Date day-of-week (BEFORE shift): "
      f"{fred_w['date'].dt.day_name().value_counts().to_dict()}")

# ── Publication lag shifts per Federal Reserve release schedule ───────────────
# FRED weekly dates are REFERENCE dates (when the measurement period ended),
# NOT publication dates. Different series have different publication lags:
#
#   Department of Labor (initial_claims, continued_claims):
#     Reference: Saturday (week-ending)
#     Publication: following Thursday at 8:30 AM
#     Shift: +5 days (Sat → Thu)
#
#   Federal Reserve H.4.1 (fed_assets, tga, reserves):
#     Reference: Wednesday (week-ending)
#     Publication: next day Thursday at 4:30 PM
#     Shift: +1 day (Wed → Thu)
#
#   Federal Reserve H.8 (bank_credit, ci_loans):
#     Reference: Wednesday (week-ending)
#     Publication: Friday of the FOLLOWING week at 4:15 PM
#     Shift: +9 days (Wed → Fri of next week)

h41_cols = ['fed_assets', 'tga', 'reserves']
h8_cols = ['bank_credit', 'ci_loans']
claims_cols = ['initial_claims', 'continued_claims']

dow = fred_w['date'].dt.dayofweek  # Mon=0 ... Sun=6

# ── Split into three groups and shift dates ──────────────────────────────────

# Claims: Saturday reference → Thursday publication (+5 days)
claims = fred_w[dow == 5][['date'] + [c for c in claims_cols if c in fred_w.columns]].copy()
claims['date'] += pd.Timedelta(days=5)
claims = claims.sort_values('date').reset_index(drop=True)

# H.4.1: Wednesday reference → Thursday publication (+1 day)
wed_mask = dow == 2
h41 = fred_w[wed_mask][['date'] + [c for c in h41_cols if c in fred_w.columns]].copy()
h41['date'] += pd.Timedelta(days=1)
h41 = h41.sort_values('date').reset_index(drop=True)

# H.8: Wednesday reference → Friday of next week publication (+9 days)
h8 = fred_w[wed_mask][['date'] + [c for c in h8_cols if c in fred_w.columns]].copy()
h8['date'] += pd.Timedelta(days=9)
h8 = h8.sort_values('date').reset_index(drop=True)

print(f"\n  After publication-date shifts:")
print(f"    Claims (Sat→Thu +5d): {len(claims):,} rows, "
      f"days: {claims['date'].dt.day_name().value_counts().to_dict()}")
print(f"    H.4.1 (Wed→Thu +1d): {len(h41):,} rows, "
      f"days: {h41['date'].dt.day_name().value_counts().to_dict()}")
print(f"    H.8 (Wed→Fri +9d):   {len(h8):,} rows, "
      f"days: {h8['date'].dt.day_name().value_counts().to_dict()}")

del fred_w

# ── Merge each group separately onto the panel ──────────────────────────────
# Three independent merge_asof calls. Each series carries forward its own
# most recent value without interfering with the others. This avoids the
# sparse-outer-join trap where NaN from one group overwrites another.

panel = panel.sort_values('date').reset_index(drop=True)
n_before = len(panel)

# Claims
panel = pd.merge_asof(panel, claims, on='date', direction='backward')
assert len(panel) == n_before, f"Row explosion after claims merge"
n_claims = panel['initial_claims'].notna().sum()
print(f"\n  Claims matched: {n_claims:,} / {len(panel):,} ({n_claims/len(panel)*100:.1f}%)")
del claims

# H.4.1
panel = pd.merge_asof(panel, h41, on='date', direction='backward')
assert len(panel) == n_before, f"Row explosion after H.4.1 merge"
n_h41 = panel['fed_assets'].notna().sum()
print(f"  H.4.1 matched: {n_h41:,} / {len(panel):,} ({n_h41/len(panel)*100:.1f}%)")
del h41

# H.8
panel = pd.merge_asof(panel, h8, on='date', direction='backward')
assert len(panel) == n_before, f"Row explosion after H.8 merge"
n_h8 = panel['bank_credit'].notna().sum()
print(f"  H.8 matched: {n_h8:,} / {len(panel):,} ({n_h8/len(panel)*100:.1f}%)")
del h8

# Build combined FRED weekly factor list
fred_w_factors = [c for c in claims_cols + h41_cols + h8_cols if c in panel.columns]
print(f"\n  FRED weekly total factors: {len(fred_w_factors)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8: COMPUTE DERIVED FACTORS
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 8: COMPUTE DERIVED FACTORS")
print("=" * 90)

derived_factors = []

# VIX futures basis = front-month futures - VIX spot
# Positive = futures premium (normal), negative = spot premium (fear)
if all(c in panel.columns for c in ['vix_fut_front', 'vix']):
    panel['vix_futures_basis'] = panel['vix_fut_front'] - panel['vix']
    derived_factors.append('vix_futures_basis')
    valid = panel['vix_futures_basis'].notna().sum()
    print(f"\n  vix_futures_basis: {valid:,} valid")
    print(f"    Mean: {panel['vix_futures_basis'].mean():.3f}")
    print(f"    Range: [{panel['vix_futures_basis'].min():.3f}, "
          f"{panel['vix_futures_basis'].max():.3f}]")
else:
    missing = [c for c in ['vix_fut_front', 'vix'] if c not in panel.columns]
    print(f"\n  ⚠ Cannot compute vix_futures_basis — missing: {missing}")

print(f"\n  Derived factors: {derived_factors}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 9: FINAL COLUMN INVENTORY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 9: FINAL COLUMN INVENTORY")
print("=" * 90)

all_factor_cols = [c for c in panel.columns if c != 'date']

print(f"\n  Factor breakdown by source:")
print(f"    FRED daily:       {len([c for c in fred_d_factors if c in panel.columns]):>4d}")
print(f"    WRDS macro daily: {len([c for c in macro_d_factors if c in panel.columns]):>4d}")
print(f"    VIX/SKEW:         {len([c for c in vix_skew_factors if c in panel.columns]):>4d}")
print(f"    CFTC:             {len([c for c in cftc_factors if c in panel.columns]):>4d}")
print(f"    AAII:             {len([c for c in aaii_factors if c in panel.columns]):>4d}")
print(f"    FRED weekly:      {len(fred_w_factors):>4d}")
print(f"    Derived:          {len(derived_factors):>4d}")
print(f"    ────────────────────")
print(f"    Total:            {len(all_factor_cols):>4d}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 10: VALIDATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 10: VALIDATION")
print("=" * 90)

# ── 10a. No duplicates ──────────────────────────────────────────────────────
n_dupes = panel['date'].duplicated().sum()
print(f"\n  Duplicate dates: {n_dupes}")
assert n_dupes == 0, f"Found {n_dupes} duplicate dates!"

# ── 10b. Shape ──────────────────────────────────────────────────────────────
print(f"\n  Total rows: {len(panel):,}")
print(f"  Total columns: {panel.shape[1]}")
print(f"  Date range: {panel['date'].min().date()} → {panel['date'].max().date()}")

# ── 10c. NaN summary by source ──────────────────────────────────────────────
print(f"\n  NaN summary by source (after forward-fill):")
for label, cols in [('FRED daily', fred_d_factors),
                     ('WRDS macro', macro_d_factors),
                     ('VIX/SKEW', vix_skew_factors),
                     ('CFTC', cftc_factors),
                     ('AAII', aaii_factors),
                     ('FRED weekly', fred_w_factors),
                     ('Derived', derived_factors)]:
    present_cols = [c for c in cols if c in panel.columns]
    if not present_cols:
        continue
    nan_rate = panel[present_cols].isna().mean().mean() * 100
    n_nan = panel[present_cols].isna().sum().sum()
    total = len(panel) * len(present_cols)
    print(f"    {label:<15s} {nan_rate:>5.2f}% avg NaN  "
          f"({n_nan:,} / {total:,} cells)")

# ── 10d. Remaining NaN — show per column for any with >0% ───────────────────
print(f"\n  Columns with remaining NaN:")
nan_per_col = panel[all_factor_cols].isna().sum()
nan_cols = nan_per_col[nan_per_col > 0].sort_values(ascending=False)
if len(nan_cols) > 0:
    print(f"\n  {'Column':<35s} {'NaN':>6s}  {'%':>6s}")
    print("  " + "-" * 50)
    for col in nan_cols.head(25).index:
        n = int(nan_cols[col])
        pct = n / len(panel) * 100
        print(f"  {col:<35s} {n:>6,d}  {pct:>5.2f}%")
    if len(nan_cols) > 25:
        print(f"  ... and {len(nan_cols) - 25} more")
else:
    print(f"  ✓ No remaining NaN")

# ── 10e. FRED weekly health check ───────────────────────────────────────────
# Verify the publication lag shifts worked — check that FRED weekly factors
# are NOT populated on days before they should be available.
print(f"\n  FRED weekly health check:")
for label, col, expected_start in [
    ('Claims (initial_claims)', 'initial_claims', None),
    ('H.4.1 (fed_assets)', 'fed_assets', None),
    ('H.8 (bank_credit)', 'bank_credit', None)
]:
    if col in panel.columns:
        first_valid = panel[panel[col].notna()]['date'].min()
        print(f"    {label}: first valid date = {first_valid.date()}")

# ── 10f. Day-of-week ────────────────────────────────────────────────────────
print(f"\n  Day-of-week distribution:")
dow = panel['date'].dt.day_name().value_counts()
print(dow.to_string())

# ── 10g. Sample ─────────────────────────────────────────────────────────────
print(f"\n  Sample (first 5 rows, selected columns):")
sample_cols = ['date', 'vix', 'mktrf', 'vix_fut_front', 'vix_term_spread',
               'skew', 'vix_futures_basis', 'initial_claims', 'fed_assets',
               'bank_credit']
sample_cols = [c for c in sample_cols if c in panel.columns]
print(panel[sample_cols].head(5).to_string(index=False))

print(f"\n  Sample (row from mid-2020):")
mid_2020 = panel[panel['date'] >= '2020-06-15'].head(1)
print(mid_2020[sample_cols].to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 11: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 11: SAVE")
print("=" * 90)

panel = panel.sort_values('date').reset_index(drop=True)

out_path = OUT_DIR / 'panel_macro_daily.parquet'
panel.to_parquet(out_path, index=False, engine='pyarrow')

print(f"\n  ✓ Saved: {out_path}")
print(f"    {len(panel):,} rows × {panel.shape[1]} columns")
print(f"    Key: date")
print(f"    Factors: {len(all_factor_cols)}")
print(f"    Size: {out_path.stat().st_size / 1e6:.1f} MB")

print("\nPanel C (macro daily) complete.")

STEP 1: ESTABLISH TRADING CALENDAR

  Trading calendar: 5,285 days
  Date range: 2004-01-02 → 2024-12-31
  Weekends: 0

STEP 2: JOIN FRED DAILY

  FRED daily: 5,254 rows × 50 factors
  Matched: 5,246 / 5,285 (99.3%)
  Forward-fill (limit=5): 3,099 → 1,152 NaN (filled 1,947)

STEP 3: JOIN WRDS MACRO DAILY

  WRDS macro daily: 5,475 rows × 42 factors
  Matched: 5,285 / 5,285 (100.0%)
  Forward-fill (limit=5): 3,128 → 19 NaN (filled 3,109)

STEP 4: JOIN VIX FUTURES + CBOE SKEW

  VIX/SKEW: 5,286 rows × 14 factors
  Matched: 5,225 / 5,285 (98.9%)
  Forward-fill (limit=5): 732 → 618 NaN (filled 114)
  (Remaining NaN is structural: VIX futures launch Mar 2004, SKEW warmup)

STEP 5: JOIN CFTC (WEEKLY — POINT-IN-TIME)

  Using available_date for point-in-time merge (6-day lag)
  CFTC: 969 rows × 22 factors
  Date range: 2006-06-19 → 2025-01-06
  Matched: 4,666 / 5,285 (88.3%)
  First valid CFTC date: 2006-06-19

STEP 6: JOIN AAII SENTIMENT (WEEKLY)

  AAII: 1,095 rows × 5 factors
  Date day-of